In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "anndata>=0.12.11",
#     "scanpy>=1.12.1",
#     "zarr>=3.1.6",
# ]
#
# [tool.uv]
# exclude-newer = "2026-05-03T06:54:44.427134817+02:00"
# ///

In [2]:
from pathlib import Path

In [3]:
import anndata as ad
import numpy as np
import scanpy as sc

In [4]:
INPUT_PATH = Path("habib17.h5ad")
RAW_OUTPUT_PATH = Path("test-data/habib17.zarr")
OUTPUT_PATH = Path("test-data/habib17-differential-expression-test-data.zarr")
GROUPBY_COLUMN = "CellType"

In [5]:
def _sample_expression_values(x: object, max_items: int = 10000) -> np.ndarray:
    if hasattr(x, "tocoo"):
        values = np.asarray(x.data)
    else:
        values = np.asarray(x).ravel()

    if values.size == 0:
        return values

    if values.size > max_items:
        step = max(1, values.size // max_items)
        values = values[::step]

    return values

In [6]:
def _needs_preprocessing(adata: ad.AnnData) -> tuple[bool, str]:
    if "log1p" in adata.uns:
        return False, "Detected adata.uns['log1p']; matrix appears already log-transformed."

    values = _sample_expression_values(adata.X)
    if values.size == 0:
        return True, "Empty matrix sample; applying preprocessing by default."

    tol = 1e-6
    non_integer_fraction = float(np.mean(np.abs(values - np.round(values)) > tol))
    max_value = float(np.max(values))

    # Heuristic: mostly non-integers with compressed range usually indicates logged data.
    if non_integer_fraction > 0.2 and max_value < 50:
        return (
            False,
            (
                "Expression values look already transformed "
                f"(non-integer fraction={non_integer_fraction:.3f}, max={max_value:.3f})."
            ),
        )

    return (
        True,
        (
            "Expression values look like raw counts "
            f"(non-integer fraction={non_integer_fraction:.3f}, max={max_value:.3f})."
        ),
    )

In [7]:
def main() -> None:
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")

    adata = ad.read_h5ad(INPUT_PATH)

    if GROUPBY_COLUMN not in adata.obs.columns:
        available = ", ".join(adata.obs.columns.astype(str).tolist())
        raise KeyError(
            f"Column '{GROUPBY_COLUMN}' not found in obs. Available columns: {available}"
        )

    # Align with the zarr v3 output style used in other dataset creation scripts.
    ad.settings.zarr_write_format = 3
    ad.settings.write_csr_csc_indices_with_min_possible_dtype = True
    ad.settings.auto_shard_zarr_v3 = True

    RAW_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    adata.write_zarr(RAW_OUTPUT_PATH)

    print(f"Wrote source AnnData to {RAW_OUTPUT_PATH}")

    # Make sure group labels are categorical for rank_genes_groups.
    adata.obs[GROUPBY_COLUMN] = adata.obs[GROUPBY_COLUMN].astype("category")

    should_preprocess, reason = _needs_preprocessing(adata)
    print(reason)

    if should_preprocess:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        print("Applied preprocessing: normalize_total + log1p")
    else:
        print("Skipped preprocessing")

    sc.tl.rank_genes_groups(
        adata,
        groupby=GROUPBY_COLUMN,
        method="wilcoxon",
        corr_method="benjamini-hochberg",
        pts=True,
        key_added="diff01",
        use_raw=False,
    )

    
    sc.tl.rank_genes_groups(
        adata,
        groupby=GROUPBY_COLUMN,
        groups=["ASC2"],
        method="wilcoxon",
        corr_method="benjamini-hochberg",
        pts=True,
        key_added="diff02",
        use_raw=False,
    )

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    adata.write_zarr(OUTPUT_PATH)

    print(f"Wrote DE results to {OUTPUT_PATH}")

In [8]:
if __name__ == "__main__":
    main()

/home/klaus/.cache/uv/environments-v2/juv-tmp-4bdktxml-2e59d8ba291d3d80/lib/python3.12/site-packages/zarr/core/array.py:4442: ZarrUserWarning: Automatic shard shape inference is experimental and may change without notice.
  shard_shape_parsed, chunk_shape_parsed = _auto_partition(
/home/klaus/.cache/uv/environments-v2/juv-tmp-4bdktxml-2e59d8ba291d3d80/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Wrote source AnnData to test-data/habib17.zarr
Expression values look already transformed (non-integer fraction=1.000, max=5.169).
Skipped preprocessing


/home/klaus/.cache/uv/environments-v2/juv-tmp-4bdktxml-2e59d8ba291d3d80/lib/python3.12/site-packages/zarr/core/array.py:4442: ZarrUserWarning: Automatic shard shape inference is experimental and may change without notice.
  shard_shape_parsed, chunk_shape_parsed = _auto_partition(
/home/klaus/.cache/uv/environments-v2/juv-tmp-4bdktxml-2e59d8ba291d3d80/lib/python3.12/site-packages/zarr/core/dtype/npy/structured.py:591: UnstableSpecificationWarning: The data type (Struct(fields=(('ASC1', FixedLengthUTF32(length=17, endianness='little')), ('ASC2', FixedLengthUTF32(length=17, endianness='little')), ('END', FixedLengthUTF32(length=17, endianness='little')), ('GABA1', FixedLengthUTF32(length=17, endianness='little')), ('GABA2', FixedLengthUTF32(length=17, endianness='little')), ('MG', FixedLengthUTF32(length=17, endianness='little')), ('NSC', FixedLengthUTF32(length=17, endianness='little')), ('ODC1', FixedLengthUTF32(length=17, endianness='little')), ('OPC', FixedLengthUTF32(length=17, endi

Wrote DE results to test-data/habib17-differential-expression-test-data.zarr


/home/klaus/.cache/uv/environments-v2/juv-tmp-4bdktxml-2e59d8ba291d3d80/lib/python3.12/site-packages/zarr/core/dtype/npy/structured.py:591: UnstableSpecificationWarning: The data type (Struct(fields=(('ASC2', FixedLengthUTF32(length=17, endianness='little')),))) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/home/klaus/.cache/uv/environments-v2/juv-tmp-4bdktxml-2e59d8ba291d3d80/lib/python3.12/site-packages/zarr/core/dtype/npy/structured.py:591: UnstableSpecificationWarning: The data type (Struct(fields=(('ASC2', Float32(endianness='little')),))) does not have a Zarr V3 specification. 

## Results

In [9]:
adata = ad.read_zarr(OUTPUT_PATH)
adata


AnnData object with n_obs × n_vars = 13067 × 5782
    obs: 'CellType', 'n_counts', 'log1p_n_counts', 'n_genes', 'log1p_n_genes', 'percent_mito', 'percent_ribo', 'percent_hb', 'percent_top50'
    var: 'gene_ids', 'mito', 'ribo', 'hb', 'n_counts', 'n_cells', 'n_genes', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'diff01', 'diff02', 'leiden', 'neighbors', 'pca'
    obsm: 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [10]:
result_set ="diff01"

In [11]:
adata.uns[result_set].keys()

dict_keys(['logfoldchanges', 'names', 'params', 'pts', 'pts_rest', 'pvals', 'pvals_adj', 'scores'])

In [12]:
adata.uns[result_set]["params"]

{'corr_method': 'benjamini-hochberg',
 'groupby': 'CellType',
 'layer': None,
 'method': 'wilcoxon',
 'reference': 'rest',
 'use_raw': False}

## Genes Expressiing

In [13]:
adata.uns[result_set]["pts"]

,ASC1,ASC2,END,GABA1,GABA2,MG,NSC,ODC1,OPC,Unclassified,exCA1,exCA3,exDG,exPFC1,exPFC2
index,,,,,,,,,,,,,,,
LINC00115,0.007128,0.007207,0.000000,0.004728,0.001277,0.000000,0.011236,0.005263,0.000000,0.002183,0.002770,0.000000,0.009279,0.004121,0.009050
RP11-54O7.1,0.003055,0.000000,0.000000,0.001182,0.002554,0.006536,0.000000,0.000752,0.003236,0.000000,0.000000,0.004511,0.000714,0.002747,0.004525
LINC02593,0.004073,0.012613,0.000000,0.000000,0.001277,0.003268,0.000000,0.001128,0.000000,0.000000,0.002770,0.000000,0.000714,0.000687,0.004525
SAMD11,0.009165,0.039640,0.000000,0.001182,0.007663,0.000000,0.000000,0.000376,0.001618,0.000000,0.000000,0.000000,0.001428,0.001030,0.000000
ISG15,0.008147,0.045045,0.123967,0.021277,0.021711,0.006536,0.033708,0.003383,0.024272,0.002183,0.024931,0.003008,0.005710,0.009272,0.018100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CTA-357J21.1,0.009165,0.003604,0.000000,0.015366,0.006386,0.009804,0.005618,0.003008,0.000000,0.006550,0.008310,0.010526,0.005710,0.010646,0.022624
RP11-28F1.2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.011236,0.000376,0.003236,0.000000,0.000000,0.000000,0.000000,0.002404,0.000000
RP11-638I8.1,0.005092,0.001802,0.000000,0.003546,0.001277,0.000000,0.016854,0.004135,0.001618,0.000000,0.002770,0.000000,0.002141,0.002747,0.004525


In [14]:
adata.var.head()

,gene_ids,mito,ribo,hb,n_counts,n_cells,n_genes,highly_variable,means,dispersions,dispersions_norm
index,,,,,,,,,,,
LINC00115,21,False,False,False,61.0,65,61,True,0.066755,3.035625,0.803781
RP11-54O7.1,24,False,False,False,25.0,34,25,True,0.023525,2.982569,0.639962
LINC02593,26,False,False,False,23.0,28,21,True,0.029433,3.274476,1.541276
SAMD11,27,False,False,False,50.0,68,45,True,0.055600,3.248546,1.461213
ISG15,34,False,False,False,192.0,243,166,True,0.172761,3.093359,0.982046


## Valeus

In [15]:
adata.uns[result_set]["pvals_adj"]

array([(2.01409853e-236, 3.32869752e-202, 7.52583736e-45, 8.15264740e-75, 1.95798612e-233, 1.33657642e-57, 1.11480581e-85, 0.00000000e+000, 5.92994405e-168, 2.28326704e-42, 2.12718245e-16, 1.22132706e-186, 1.27506013e-48, 1.05903487e-046, 7.94606500e-09),
       (1.08977655e-206, 2.60567505e-172, 3.60472397e-44, 1.02031573e-64, 2.94542503e-167, 1.51568302e-44, 3.14072452e-81, 0.00000000e+000, 3.01554803e-160, 1.88408607e-36, 6.69216913e-10, 5.37992054e-091, 2.70066637e-27, 3.08407480e-041, 1.83592661e-08),
       (7.15411945e-179, 1.26441774e-142, 1.08835200e-37, 1.44138319e-58, 5.14291787e-147, 2.20116892e-42, 7.09577296e-73, 0.00000000e+000, 1.21709051e-135, 1.88408607e-36, 2.63390866e-08, 5.37992054e-091, 4.15695472e-27, 5.79607902e-034, 2.38774530e-07),
       ...,
       (4.77390694e-032, 3.23100658e-012, 1.52989287e-04, 1.68707958e-20, 4.22338348e-017, 5.65959368e-14, 3.88408190e-05, 1.05584031e-062, 2.23023699e-013, 7.75410212e-09, 2.73077038e-05, 3.37295310e-015, 8.89093347e-37

In [16]:
adata.uns[result_set]["logfoldchanges"]

array([( 5.1148477,  5.8159356,  9.125652 ,  2.1814058,  6.7434726,  7.624139 ,  7.126465 ,  4.6206083,  5.381047 ,  3.430128 ,  1.890498 ,  5.431732 ,  3.2687201,  1.8852141,  1.4863284),
       ( 3.0355082,  4.308961 ,  6.2816014,  3.9494898,  4.5878334,  8.129015 ,  9.348767 ,  6.074401 ,  6.388594 ,  3.2173357,  1.9080889,  3.1544108,  2.2362506,  0.9926136,  1.4648893),
       ( 4.5896244,  5.504424 ,  6.9052052,  6.9844966,  6.771658 ,  7.3661633,  7.1575685,  5.307813 ,  5.4136667,  3.3086011,  1.2360407,  2.630762 ,  2.2186584,  2.5081437,  1.3753058),
       ...,
       (-1.5001757, -1.6717236, -2.390654 , -1.88315  , -1.7524157, -2.869834 , -1.3499839, -1.8437475, -2.3154173, -2.0766509, -1.7159787, -2.1019545, -2.581842 , -3.5846999, -1.904816 ),
       (-3.0981302, -3.1463888, -3.6965976, -2.8949077, -2.5319786, -3.1130056, -2.605768 , -3.15221  , -3.296601 , -1.8668375, -1.9583945, -2.5071673, -3.5166638, -3.1270483, -1.5541416),
       (-3.3740058, -3.2507138, -2.9065025,

In [17]:
adata.uns[result_set]["logfoldchanges"]

array([( 5.1148477,  5.8159356,  9.125652 ,  2.1814058,  6.7434726,  7.624139 ,  7.126465 ,  4.6206083,  5.381047 ,  3.430128 ,  1.890498 ,  5.431732 ,  3.2687201,  1.8852141,  1.4863284),
       ( 3.0355082,  4.308961 ,  6.2816014,  3.9494898,  4.5878334,  8.129015 ,  9.348767 ,  6.074401 ,  6.388594 ,  3.2173357,  1.9080889,  3.1544108,  2.2362506,  0.9926136,  1.4648893),
       ( 4.5896244,  5.504424 ,  6.9052052,  6.9844966,  6.771658 ,  7.3661633,  7.1575685,  5.307813 ,  5.4136667,  3.3086011,  1.2360407,  2.630762 ,  2.2186584,  2.5081437,  1.3753058),
       ...,
       (-1.5001757, -1.6717236, -2.390654 , -1.88315  , -1.7524157, -2.869834 , -1.3499839, -1.8437475, -2.3154173, -2.0766509, -1.7159787, -2.1019545, -2.581842 , -3.5846999, -1.904816 ),
       (-3.0981302, -3.1463888, -3.6965976, -2.8949077, -2.5319786, -3.1130056, -2.605768 , -3.15221  , -3.296601 , -1.8668375, -1.9583945, -2.5071673, -3.5166638, -3.1270483, -1.5541416),
       (-3.3740058, -3.2507138, -2.9065025,

## Genes

In [18]:
adata.uns[result_set]["names"][5781]

np.void(('TTLL7', 'TTLL7', 'TTLL7', 'GPM6B', 'GPM6B', 'TTLL7', 'PCDH9', 'SPARCL1', 'TTLL7', 'BAZ2B', 'SPARCL1', 'NEAT1', 'SPARCL1', 'MBP', 'NEAT1'), dtype=[('ASC1', 'O'), ('ASC2', 'O'), ('END', 'O'), ('GABA1', 'O'), ('GABA2', 'O'), ('MG', 'O'), ('NSC', 'O'), ('ODC1', 'O'), ('OPC', 'O'), ('Unclassified', 'O'), ('exCA1', 'O'), ('exCA3', 'O'), ('exDG', 'O'), ('exPFC1', 'O'), ('exPFC2', 'O')])

In [19]:
adata.obs['CellType'].unique().astype(str)

array(['exCA1', 'exCA3', 'ASC1', 'GABA1', 'ODC1', 'exDG', 'Unclassified',
       'exPFC2', 'GABA2', 'END', 'exPFC1', 'MG', 'ASC2', 'OPC', 'NSC'],
      dtype='<U12')

In [20]:
adata.uns[result_set]["names"]['ASC2'][0:5]
for ct in adata.obs['CellType'].unique().astype(str):
    if ct in adata.uns[result_set]["names"].dtype.names:
        print(f"Top 5 Genes of {ct}: {adata.uns[result_set]['names'][ct][0:5]}")

Top 5 Genes of exCA1: ['NEFM' 'RYR3' 'KCNQ1OT1' 'OGFRL1' 'CPE']
Top 5 Genes of exCA3: ['COL5A2' 'RP11-161M6.2' 'KCNQ1OT1' 'TPM2' 'NEFM']
Top 5 Genes of ASC1: ['SLC1A2' 'MACF1' 'SLC1A3' 'ATP1A2' 'NEAT1']
Top 5 Genes of GABA1: ['SPARCL1' 'GRIK1' 'TAC1' 'SLC6A1' 'ERBB4']
Top 5 Genes of ODC1: ['MBP' 'TF' 'MOBP' 'CERCAM' 'TTLL7']
Top 5 Genes of exDG: ['SEMA5A' 'DGKG' 'FAM118A' 'CFLAR' 'RNA28S5']
Top 5 Genes of Unclassified: ['MT-ATP6' 'MT-CYB' 'MT-ND5' 'NEFM' 'MT-ND2']
Top 5 Genes of exPFC2: ['SPARCL1' 'KCNQ1OT1' 'CEP290' 'PCDH9' 'DENND3']
Top 5 Genes of GABA2: ['DLX6-AS1' 'RGS12' 'CXCL14' 'SLC6A1' 'SCG2']
Top 5 Genes of END: ['CLDN5' 'B2M' 'HLA-E' 'A2M' 'XAF1']
Top 5 Genes of exPFC1: ['CCK' 'SPARCL1' 'DENND3' 'KCNQ1OT1' 'CEP290']
Top 5 Genes of MG: ['CD74' 'ADAM28' 'LPAR6' 'RNASET2' 'NEAT1']
Top 5 Genes of ASC2: ['GFAP' 'CLU' 'AQP4' 'SPARCL1' 'MACF1']
Top 5 Genes of OPC: ['PTPRZ1' 'VCAN' 'TNR' 'BCAN' 'SMOC1']
Top 5 Genes of NSC: ['DNAAF1' 'SPAG17' 'CFAP43' 'CAPS' 'ZFP36L1']


In [21]:
for ct in adata.obs['CellType'].unique().astype(str):
    if ct in adata.uns[result_set]["names"].dtype.names:
        gene = adata.uns[result_set]["names"][ct][0]
        pvalue_adj = adata.uns[result_set]["pvals_adj"][ct][0]
        logfoldchange = adata.uns[result_set]["logfoldchanges"][ct][0]
        pts = adata.uns[result_set]["pts"][ct][0]
        print(f"Top  genes of {ct}: {gene} with adjusted p value of {pvalue_adj}, logfoldchange of {logfoldchange}, and pts of {pts}")


Top  genes of exCA1: NEFM with adjusted p value of 2.127182448401547e-16, logfoldchange of 1.8904980421066284, and pts of 0.002770083102493075
Top  genes of exCA3: COL5A2 with adjusted p value of 1.2213270586106685e-186, logfoldchange of 5.431732177734375, and pts of 0.0
Top  genes of ASC1: SLC1A2 with adjusted p value of 2.0140985294020146e-236, logfoldchange of 5.114847660064697, and pts of 0.007128309572301426
Top  genes of GABA1: SPARCL1 with adjusted p value of 8.152647403913725e-75, logfoldchange of 2.181405782699585, and pts of 0.004728132387706856
Top  genes of ODC1: MBP with adjusted p value of 0.0, logfoldchange of 4.620608329772949, and pts of 0.005263157894736842
Top  genes of exDG: SEMA5A with adjusted p value of 1.2750601254111867e-48, logfoldchange of 3.2687201499938965, and pts of 0.009279086366880799
Top  genes of Unclassified: MT-ATP6 with adjusted p value of 2.2832670412486687e-42, logfoldchange of 3.4301280975341797, and pts of 0.002183406113537118
Top  genes of exP

/tmp/ipykernel_368100/1818202400.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pts = adata.uns[result_set]["pts"][ct][0]
